# Python Sales Data Analysis

**Project type:** Exploratory Data Analysis (EDA)  
**Tools:** Python, Pandas, Matplotlib  
**Dataset:** `PQ Data.csv`

## Project Overview
This project explores retail sales records to understand sales performance, product categories, profitability, customer characteristics, and geographic and time-based patterns.

## Objectives
- Inspect and prepare the dataset for analysis.
- Calculate sales, cost, profit, and profit margin.
- Compare product and sub-category performance.
- Explore sales by country and state.
- Review customer age and gender distributions.
- Analyze sales trends over time.
- Create visualizations and export an enriched dataset.

> **Data note:** The source file contains transaction-level records but does not provide a unique transaction or customer identifier. Therefore, this analysis reports record counts and quantities rather than claiming a distinct number of orders or customers.

## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("PQ Data.csv")
print("Rows and columns:", data.shape)
display(data.head())

## 2. Explore the data

In [ ]:
display(data.info())
display(data.describe(include="all"))
display(data.isna().sum())

## 3. Data cleaning

In [ ]:
# Convert dates to datetime
data["Date"] = pd.to_datetime(data["Date"], errors="coerce")

# Remove currency symbols and convert prices/costs to numeric values
for column in ["Unit Cost", "Unit Price"]:
    data[column] = pd.to_numeric(
        data[column].astype(str).str.replace(r"[$,]", "", regex=True).str.strip(),
        errors="coerce"
    )

required_columns = [
    "Date", "Customer Age", "Customer Gender", "Country", "State",
    "Product Category", "Sub Category", "Quantity", "Unit Cost", "Unit Price"
]
print("Missing values after conversion:")
display(data.isna().sum())

before = len(data)
data = data.dropna(subset=required_columns).copy()
print(f"Rows retained: {len(data):,} of {before:,}")

## 4. Feature engineering

In [ ]:
data["Total Sales"] = data["Quantity"] * data["Unit Price"]
data["Total Cost"] = data["Quantity"] * data["Unit Cost"]
data["Profit"] = data["Total Sales"] - data["Total Cost"]
data["Profit Margin"] = data["Profit"].div(data["Total Sales"].replace(0, pd.NA))

data["Year"] = data["Date"].dt.year
data["Month"] = data["Date"].dt.month
data["Month Name"] = data["Date"].dt.strftime("%b")
data["Quarter"] = data["Date"].dt.to_period("Q").astype(str)

display(data.head())

## 5. Key performance indicators

In [ ]:
total_sales = data["Total Sales"].sum()
total_cost = data["Total Cost"].sum()
total_profit = data["Profit"].sum()
total_units = data["Quantity"].sum()
overall_profit_margin = total_profit / total_sales if total_sales else 0

kpis = pd.DataFrame({
    "Metric": ["Total Sales", "Total Cost", "Total Profit", "Total Units Sold", "Overall Profit Margin"],
    "Value": [total_sales, total_cost, total_profit, total_units, overall_profit_margin]
})
display(kpis)

## 6. Product performance analysis

In [ ]:
category_performance = (
    data.groupby("Product Category")
    .agg(
        Total_Sales=("Total Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Units_Sold=("Quantity", "sum")
    )
    .sort_values("Total_Sales", ascending=False)
)
display(category_performance)

subcategory_performance = (
    data.groupby("Sub Category")
    .agg(Total_Sales=("Total Sales", "sum"), Total_Profit=("Profit", "sum"))
    .sort_values("Total_Sales", ascending=False)
)
display(subcategory_performance.head(10))

## 7. Geographic analysis

In [ ]:
country_performance = (
    data.groupby("Country")
    .agg(Total_Sales=("Total Sales", "sum"), Total_Profit=("Profit", "sum"))
    .sort_values("Total_Sales", ascending=False)
)
display(country_performance)

top_states = data.groupby("State")["Total Sales"].sum().sort_values(ascending=False).head(10)
display(top_states)

## 8. Customer profile analysis

In [ ]:
display(data["Customer Age"].describe().to_frame("Customer Age"))
gender_distribution = data["Customer Gender"].value_counts()
display(gender_distribution.to_frame("Record Count"))

## 9. Time-based analysis

In [ ]:
monthly_sales = data.set_index("Date").resample("MS")["Total Sales"].sum()
yearly_sales = data.groupby("Year")["Total Sales"].sum()

display(yearly_sales.to_frame("Total Sales"))
display(monthly_sales.sort_values(ascending=False).head(10).to_frame("Total Sales"))

## 10. Visualizations

In [ ]:
category_performance["Total_Sales"].sort_values().plot(
    kind="barh", title="Total Sales by Product Category", figsize=(8, 5)
)
plt.xlabel("Total Sales")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()

In [ ]:
monthly_sales.plot(figsize=(11, 5), marker="o", title="Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

In [ ]:
gender_distribution.plot(kind="bar", title="Records by Customer Gender", figsize=(6, 4))
plt.xlabel("Customer Gender")
plt.ylabel("Number of Records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Export the prepared dataset

The enriched dataset includes calculated sales, cost, profit, profit margin, and date-related fields.

In [ ]:
data.to_csv("cleaned_sales_analysis.csv", index=False)
print("Export complete: cleaned_sales_analysis.csv")

## Conclusion

This analysis demonstrates a complete introductory data-analysis workflow: loading data, inspecting data quality, converting data types, creating analytical features, grouping and aggregating records, and communicating patterns through visualizations.

**Important interpretation note:** Sales and profit are calculated at the record level using `Quantity × Unit Price` and `Quantity × Unit Cost`. The dataset does not include a unique order ID, so order-level metrics such as average order value are not calculated.